# Wehrl entropy and IPR of the memory under the ATS pump (poster parameters)

A two-photon-pumped ATS (memory `a`, buffer `b`). The memory starts in the cat
`|i alpha> + |-i alpha>`, `alpha = sqrt(epsilon_d / (2 g2))`, and the buffer
in vacuum. Two measures of how much the memory state spreads are computed on
its reduced state `rho_a = Tr_b rho`, in the lab frame and the Fock basis:

- the **Wehrl entropy** `S_W`, from the Husimi Q function, which is obtained by
  Gaussian smoothing of the Wigner function, at `t = 0` and `t = T`;
- the **inverse participation ratio** in the Fock basis,
  `IPR = sum_k |<k|psi>|^4`. `rho_a` is mixed, so this is evaluated as
  `IPR = sum_k <k|rho_a|k>^2 = sum_k p_k^2`, which is the same thing for a pure
  state. It is computed at every point of `tsave`. `IPR = 1` for a single Fock
  state and about `1 / N` for a state spread evenly over `N` of them.

## The interaction frame of Floquet-Lindblad

With `w_b = 2 w_a` and the drive at `w_d = w_b`, the pump frequency
`w_p = 2 w_a - w_d` of the original notebooks is 0. The pump term
`-2 E_J sin(epsilon_0 cos(w_p t))` is then the constant
`-2 E_J sin(epsilon_0)`, and the system is exactly Floquet-Lindblad's
full-ATS model (`Floquet-Lindblad/docs/models_ats.md`):

    H = H_s + epsilon_d cos(w_b t) (b + b^dag),
    H_s = w_a a^dag a + w_b b^dag b - 2 E_J sin(epsilon_0) [sin(phi_a + phi_b) - phi_a - phi_b],

with `phi_a = 0.11 (a + a^dag)`, `phi_b = 0.204 (b + b^dag)`, the same as the
`varphi` operators here. The simulation runs in Floquet-Lindblad's
interaction frame of `H_s` (`build_ats_hamiltonian_interaction`):

- `H_s = V diag(lam) V^dag` is diagonalized once, and the state is written in
  its eigenbasis;
- the only Hamiltonian left is the drive, `H_I(t) = epsilon_d cos(w_b t)
  (b~(t) + b~(t)^dag)` with `b~(t) = e^{i H_s t} b e^{-i H_s t}`. The buffer
  loss becomes `sqrt(kappa) b~(t)`.

This removes the large static part of `H` (`w_a`, `w_b` and the Josephson
term) from the integration. To get the lab-frame state back at time `t`, the
notebook applies `rho = V e^{-i lam t} rho_I e^{i lam t} V^dag`, then traces
out the buffer.

The builders are imported from the Floquet-Lindblad checkout, which is shipped
to the workers (`py_modules`). The physical constants are passed in
explicitly, and they are the poster ones, not Floquet-Lindblad's defaults:


Importing the builders switches jax to x64 and dynamiqs to double precision.
The interaction frame needs this: the phases `(lam_j - lam_k) t` reach
`~10^3` rad at `T`, which single precision cannot resolve. The run is
therefore in double precision throughout, where the poster notebook used single.

## Parameters

The initial cat is normalized (`dq.unit`): `(|i alpha> + |-i alpha>) / sqrt(2)` has norm
squared `1 + exp(-2 alpha^2)`, which is `1.14` at `epsilon_d / g2 = 2`.

At most `N_MAX_GPUS` GPUs are in use at once, as in
`OTOCS_poster_init_coherent.ipynb`.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`).
   These must match the cluster.
2. **Vault session**, once (lasts about a week):

   ```bash
   export VAULT_ADDR=https://vault.int.alice-bob.com
   vault login -method=oidc role=research_core_user
   ```

3. Run all cells. The first `acr.run` of a session may open an interactive
   Microsoft device-code login for GCS read access. That is expected, and the
   login is cached.

If you have already run something that imports `ray` in this kernel, restart it
first. Ray caches `RAY_AUTH_MODE` at first import.

The GPU image's `prod-gpu` environment already has jax (CUDA), dynamiqs 0.3.6
and scipy. Only `floquet_lindblad` is shipped. Plain `dq.mesolve` is enough
here, so the patched dynamiqs that Floquet-Lindblad's solvers need (for
`mesolve_fast`) is not shipped.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import ray

from anb_compute import ray as acr

print("python", sys.version.split()[0], "| ray", ray.__version__)

In [ ]:
import pathlib


def _find_floquet_lindblad():
    """The `floquet_lindblad` package of the Floquet-Lindblad checkout.

    Notebooks have no __file__, so search upward from the working directory,
    which VS Code sets to either the notebook's folder or the workspace root.
    $FLOQUET_LINDBLAD_SRC wins.
    """
    if os.environ.get("FLOQUET_LINDBLAD_SRC"):
        return pathlib.Path(os.environ["FLOQUET_LINDBLAD_SRC"])
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


# Shipped by path, not by name, so local edits reach the workers. It prepends to
# the workers' sys.path; child tasks inherit the runtime_env of the job.
FL_PKG = _find_floquet_lindblad()
RUNTIME_ENV = acr.build_runtime_env(py_modules=[str(FL_PKG)])
print("shipping", FL_PKG)

In [ ]:
import numpy as np

# --- constants (computed once), as in OTOCS_poster ---
ħ = 1
e = 1
phi_quantum = 1/2

w_a = 25.338776456203686 #GHz
w_b = 2*w_a


phi_a = 0.11
Z_a = 2 * phi_quantum**2 * phi_a**2
C_a_inv = Z_a * w_a
E_Ca = C_a_inv / 2

phi_b = 0.204
Z_b = 2 * phi_quantum**2 * phi_b**2
C_b_inv = Z_b * w_b
E_Cb = C_b_inv / 2

E_J = 37.12 * 2 * np.pi * 1.5
kappa = 2*np.pi/10.4

# No PRECISION switch: importing floquet_lindblad's builders sets double
# precision, which the interaction frame needs.

### One point, on one GPU worker

`wehrl_ipr_point` builds the model with Floquet-Lindblad's builders,
evolves the memory and buffer with `dq.mesolve`, and returns, for the memory:

- `S_W(0)` and `S_W(T)`, with the FFT Husimi and Wehrl helpers of
  `wehlr_entropy.ipynb`, unchanged. The Wigner grid is `dq.wigner`'s default
  (`xmax = 6`, 201 pixels, `g = 2`).
- The Fock populations `p_k(t) = <k|rho_a(t)|k>` and `IPR(t) = sum_k p_k(t)^2`
  on all of `tsave`. `p_k` is divided by `Tr rho_a`, which only absorbs the
  solver's drift in the trace.

The solver is `Tsit5` at `rtol = 1e-8`, `atol = 1e-10`. At dynamiqs'
defaults (`1e-6`, `1e-8`) the two frames differ by `~1e-4` on `rho_a(T)`,
and at `1e-8` by `~1e-8`, so the defaults are not enough for 4 digits.

`frame="interaction"` is the default. `frame="lab"` integrates the same model
in the lab frame (Floquet-Lindblad's `build_ats_hamiltonian`), with the same
constants. It is there for the check below.

In the interaction frame, every saved state is turned back into a lab-frame
Fock-basis state before the trace, `rho = V e^{-i lam t} rho_I e^{i lam t}
V^dag`. `lam` is `diag(V^dag H_s V)`, with Floquet-Lindblad's own `V` and
`H_s`, the lab Hamiltonian at `epsilon_d = 0`. The full states are saved for
that (`50` of them, `0.4 GB` at `n_a = 38`, `n_b = 19`), and only `rho_a`
leaves the GPU.

In [ ]:
def wehrl_ipr_point(*, label, epsilon_0, ratio=10.0, n_a, n_b, T, n_time=50, g=2.0,
                    frame="interaction", rtol=1e-8, atol=1e-10):
    """S_W(0), S_W(T) and IPR(t) of the memory for one (epsilon_0, epsilon_d / g2), on one GPU."""
    import time

    import jax
    import jax.numpy as jnp
    import numpy as np

    # Shipped from the Floquet-Lindblad checkout. Importing it switches on
    # jax_enable_x64 and dq.set_precision("double").
    from floquet_lindblad.models.ats import (
        build_ats_hamiltonian,
        build_ats_hamiltonian_interaction,
    )

    import dynamiqs as dq

    t_start = time.time()

    # ---------- the model, from Floquet-Lindblad, with the poster constants ----------
    # Static pump (w_p = 0): sin(epsilon_0), not J_1(epsilon_0). This is Floquet-Lindblad's g.
    g2 = np.sin(epsilon_0) * E_J * phi_a**2 * phi_b
    epsilon_d = ratio * g2
    model = dict(n_a=n_a, n_b=n_b, w_a=w_a, kappa_b=kappa, E_J=E_J,
                 phi_a=phi_a, phi_b=phi_b, epsilon_p=epsilon_0)
    # Floquet-Lindblad sets epsilon_d = 2 alpha_sq g.
    alpha_sq = epsilon_d / (2 * g2)

    alpha = jnp.sqrt(epsilon_d / (2*g2))
    # normalized: (|i alpha> + |-i alpha>) / sqrt(2) has norm^2 1 + exp(-2 alpha^2)
    init_mode_a = dq.unit(dq.coherent(n_a, 1j*alpha) + dq.coherent(n_a, 1j*-alpha))
    psi_0 = dq.tensor(init_mode_a, dq.fock(n_b, 0))

    tsave = jnp.linspace(0, T, n_time)
    # Tighter than dynamiqs' defaults (1e-6, 1e-8), at which rho_a(T) is only good to ~1e-4.
    method = dq.method.Tsit5(rtol=rtol, atol=atol)

    if frame == "interaction":
        H_I, jumps_I, _, _, V, _, params = build_ats_hamiltonian_interaction(
            alpha_sq=alpha_sq, **model)
        # Its jump operators are [sqrt(kappa_1) a~, sqrt(kappa_b) b~]: keep the
        # buffer loss sqrt(2 kappa) b~ only, as in the lab-frame notebooks.
        jumps = jumps_I[1:]

        # The static Hamiltonian is the lab one without the drive.
        H_s = build_ats_hamiltonian(alpha_sq=0.0, **model)[0](0.0).to_numpy()
        lam = np.real(np.einsum("ji,jk,ki->i", V.conj(), H_s, V))
        V_j = jnp.asarray(V)

        psi_0_I = dq.asqarray(V_j.conj().T @ psi_0.to_jax(), dims=(n_a, n_b))
        result = dq.mesolve(H_I, jumps, psi_0_I, tsave, method=method)

        @jax.jit
        def to_lab_memory(rho_I, t):
            """Tr_b of V e^{-i lam t} rho_I e^{i lam t} V^dag."""
            ph = jnp.exp(-1j * jnp.asarray(lam) * t)
            rho = V_j @ (ph[:, None] * rho_I * ph.conj()[None, :]) @ V_j.conj().T
            return jnp.einsum("ibjb->ij", rho.reshape(n_a, n_b, n_a, n_b))

        states = dq.to_jax(result.states)
        state_a = jnp.stack([to_lab_memory(states[i], tsave[i]) for i in range(n_time)])
    elif frame == "lab":
        H, jumps_lab, _, params = build_ats_hamiltonian(alpha_sq=alpha_sq, **model)
        jumps = jumps_lab[1:]  # buffer loss only, as above
        save_extra = lambda rho: {"state_a": dq.ptrace(rho, 0)}
        result = dq.mesolve(H, jumps, psi_0, tsave, save_extra=save_extra, save_states=False,
                             method=method)
        state_a = dq.to_jax(result.extra["state_a"])
    else:
        raise ValueError(f"unknown frame {frame!r}")
    assert np.isclose(params["epsilon_d"], epsilon_d), (params["epsilon_d"], epsilon_d)

    # ---------- FFT-based linear convolution (mode="same") ----------
    @jax.jit
    def _fftconvolve_same_2d(W, K):
        """
        Linear convolution using FFT, returning the "same" central crop.
        W, K: (nx, ny) real arrays (same shape).
        """
        nx, ny = W.shape
        fx, fy = 2 * nx - 1, 2 * ny - 1  # full conv size

        # rFFT over full size (real -> complex half-spectrum)
        FW = jnp.fft.rfftn(W, s=(fx, fy))
        FK = jnp.fft.rfftn(K, s=(fx, fy))
        full = jnp.fft.irfftn(FW * FK, s=(fx, fy))

        # "same" crop (matches scipy.signal.fftconvolve(..., mode="same") center crop)
        sx = (nx - 1) // 2
        sy = (ny - 1) // 2
        return full[sx:sx + nx, sy:sy + ny]

    # ---------- Precompute Gaussian kernel (and reuse) ----------
    def make_husimi_from_wigner(xvec, pvec, g=2.0):
        """
        Returns a JAX-jittable function husimi(W) that computes Q from W via Gaussian convolution.
        xvec, pvec should be 1D jax arrays with uniform spacing.
        """
        xvec = jnp.asarray(xvec)
        pvec = jnp.asarray(pvec)

        dx = xvec[1] - xvec[0]
        dp = pvec[1] - pvec[0]

        # center at 0 for the kernel
        X = xvec - xvec[xvec.shape[0] // 2]
        P = pvec - pvec[pvec.shape[0] // 2]
        XX, PP = jnp.meshgrid(X, P, indexing="ij")

        K = (g**2 / (2 * jnp.pi)) * jnp.exp(-(g**2 / 2) * (XX**2 + PP**2))

        @jax.jit
        def husimi(W):
            W = jnp.asarray(W)
            Q = dx * dp * _fftconvolve_same_2d(W, K)
            return jnp.maximum(Q, 0.0)

        return husimi

    # ---------- Wehrl entropy on GPU ----------
    @jax.jit
    def wehrl_entropy_from_Q(Q, xvec, pvec, eps=1e-45):
        xvec = jnp.asarray(xvec)
        pvec = jnp.asarray(pvec)
        dx = xvec[1] - xvec[0]
        dp = pvec[1] - pvec[0]

        Qc = jnp.clip(Q, eps, None)
        # In single precision 1e-45 is subnormal and XLA flushes it to 0, so the
        # clip leaves exact zeros and 0 * log(0) = nan. Take 0 log 0 = 0 instead
        # (the clip's own contribution, ~1e-43 per pixel, is negligible anyway).
        return -jnp.sum(jnp.where(Qc > 0, Qc * jnp.log(Qc), 0.0)) * dx * dp

    def wehrl(rho):
        xvec, pvec, wigner_func = dq.wigner(rho)
        Q = make_husimi_from_wigner(xvec, pvec, g=g)(wigner_func)
        return float(wehrl_entropy_from_Q(Q, xvec, pvec))

    S_begin = wehrl(state_a[0])
    S_end = wehrl(state_a[-1])

    # ---------- Fock populations and IPR = sum_k <k|rho_a|k>^2 ----------
    pop = np.real(np.diagonal(np.asarray(state_a), axis1=-2, axis2=-1)).astype(float)
    pop = pop / pop.sum(axis=-1, keepdims=True)
    ipr = (pop**2).sum(axis=-1)

    return {
        "label": label,
        "frame": frame,
        "epsilon_0": float(epsilon_0),
        "ratio": float(ratio),
        "g2": float(g2),
        "epsilon_d": float(epsilon_d),
        "alpha": float(alpha),
        "n_a": n_a,
        "n_b": n_b,
        "T": float(T),
        "tsave": np.asarray(tsave),
        "S_begin": S_begin,
        "S_end": S_end,
        "rel_change": (S_end - S_begin) / S_begin,
        "pop": pop,
        "ipr": ipr,
        "ipr_begin": float(ipr[0]),
        "ipr_end": float(ipr[-1]),
        "state_a_end": np.asarray(state_a[-1]),
        "device": str(jax.devices()[0]),
        "elapsed_s": time.time() - t_start,
    }

### The coordinator

One job per section, one GPU task per point. At most `N_MAX_GPUS` tasks, so at
most `N_MAX_GPUS` GPUs, are in flight at once: the coordinator submits the first
`N_MAX_GPUS` points, then the next one each time a task finishes. Pass
`n_max=...` to `run_on_cluster` to override it for one call. The coordinator
only fans out and gathers, so it is submitted with `entrypoint_num_cpus=0`, which keeps it on the
head node rather than on a freshly started spot worker (see
`/workspace/ray_driver_placement.md`). If that ever misbehaves, drop the
argument: the job then runs exactly as `acr.run`'s default.

The job carries `RUNTIME_ENV`, so every task can import `floquet_lindblad`.

Each point is caught on its own: a task lost to the infrastructure (node
preempted, OOM) comes back as an `"error"` entry instead of sinking the rest.

In [ ]:
import time

# Per GPU task. One whole GPU each: jax preallocates most of the card's memory.
TASK_RESOURCES = dict(num_gpus=1, num_cpus=2)
# At most this many tasks in flight at once, i.e. at most this many GPUs taken on
# the cluster. The other points wait in the coordinator, not in Ray's queue.
N_MAX_GPUS = 32


def sweep(worker, points, common, resources, n_max):
    """Coordinator: one GPU task per point, at most `n_max` running at a time.

    `worker` is passed in because it is defined in this notebook and ships by
    value. Results come back in the order of `points`.
    """
    import ray

    from anb_compute import ray as acr

    n_max = max(1, int(n_max))
    out = [None] * len(points)
    next_i = 0
    refs = {}  # in-flight task -> its index in points
    while next_i < len(points) or refs:
        while next_i < len(points) and len(refs) < n_max:
            refs[acr.submit(worker, **resources, **common, **points[next_i])] = next_i
            next_i += 1
        ready, _ = ray.wait(list(refs), num_returns=1)
        i = refs.pop(ready[0])
        try:
            res = ray.get(ready[0])
        except Exception as exc:  # a task lost to infra, not to numerics
            res = {**points[i], "error": f"{type(exc).__name__}: {exc}"}
        out[i] = res
        print(f"{res['label']:<28} "
              + (res["error"] if "error" in res else f"{res['elapsed_s']:.1f} s"))
    return out


def run_on_cluster(worker, points, common, n_max=N_MAX_GPUS):
    t0 = time.time()
    results = acr.run(
        sweep, worker, points, common, TASK_RESOURCES, n_max,
        runtime_env=RUNTIME_ENV,  # floquet_lindblad, for every task
        poll_seconds=5,
        entrypoint_num_cpus=0,  # coordinator on the head, never on a spot node
    )
    print(f"\nwall clock, incl. job startup: {time.time() - t0:.1f} s "
          f"for {len(results)} point(s)")
    for r in results:
        if "error" in r:
            print(f"FAILED {r['label']}: {r['error']}")
    return results

# Chaotic and stabilized regimes

As in `OTOCS_poster.ipynb`: `epsilon_0 = 0.93` (chaotic) and `epsilon_0 = 0.2`
(stabilized), at `epsilon_d = 10 g2` (cat amplitude `sqrt(5)`), `n_a = 38`,
`n_b = 19`, `T = 0.2 / kappa`, `n_time = 50`, in one job.

With `CHECK_LAB_FRAME = True`, the same two points also run in the lab frame,
in the same job. The two frames must agree to the solver tolerance, about
`1e-8` here.

In [ ]:
import matplotlib.pyplot as plt

T = 2 / kappa
n_time = 50
n_a = 38
n_b = (n_a + 1) // 2

CHECK_LAB_FRAME = True

REGIMES = [
    dict(label="chaotic, eps_0 = 0.93", epsilon_0=0.93),
    dict(label="stabilized, eps_0 = 0.2", epsilon_0=0.2),
]
LAB_CHECK = [dict(r, label=r["label"] + ", lab", frame="lab") for r in REGIMES]
COMMON = dict(ratio=10.0, n_a=n_a, n_b=n_b, T=T, n_time=n_time)

out = run_on_cluster(wehrl_ipr_point, REGIMES + (LAB_CHECK if CHECK_LAB_FRAME else []), COMMON)
regimes, regimes_lab = out[:len(REGIMES)], out[len(REGIMES):]
print()
for res in regimes + regimes_lab:
    if "error" not in res:
        print(f"{res['label']:<34} S_W(0) = {res['S_begin']:.6g}   S_W(T) = {res['S_end']:.6g}   "
              f"(S_W(T) - S_W(0)) / S_W(0) = {res['rel_change']:.6g}")
        print(f"{'':<34} IPR(0) = {res['ipr_begin']:.6g}   IPR(T) = {res['ipr_end']:.6g}   "
              f"{res['elapsed_s']:.1f} s on {res['device']}")

In [ ]:
# Interaction frame vs lab frame, point by point.
for res, lab in zip(regimes, regimes_lab):
    if "error" in res or "error" in lab:
        print(f"{res['label']}: a run failed, no comparison")
        continue
    print(f"{res['label']:<28} max |IPR_I - IPR_lab| = {np.max(np.abs(res['ipr'] - lab['ipr'])):.2e}   "
          f"|S_W,I(T) - S_W,lab(T)| = {abs(res['S_end'] - lab['S_end']):.2e}   "
          f"max |rho_a,I(T) - rho_a,lab(T)| = {np.max(np.abs(res['state_a_end'] - lab['state_a_end'])):.2e}")

In [ ]:
ok_regimes = [r for r in regimes if "error" not in r]

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True)

ax = axes[0]
for res in ok_regimes:
    ax.plot(res["tsave"] * kappa, res["ipr"], label=res["label"])
ax.set_ylabel(r'IPR $=\sum_k \langle k|\rho_a|k\rangle^2$')
ax.set_xlabel(r'$\kappa_b t$')
ax.set_title(r'$n_a=$'+str(n_a)+', 'r'$n_b=$'+str(n_b))
ax.legend(fontsize=8)

ax = axes[1]
k = np.arange(n_a)
ax.bar(k, ok_regimes[0]["pop"][0], color="0.75", label=r"$t=0$")
for res in ok_regimes:
    ax.plot(k, res["pop"][-1], ".-", label=f"{res['label']}, " + r"$t=T$")
ax.set_ylabel(r'$p_k = \langle k|\rho_a|k\rangle$')
ax.set_xlabel(r'Fock state $k$')
ax.set_title(r'Fock populations, $T = 0.2/\kappa_b$')
ax.legend(fontsize=8)
plt.show()

# Heatmap

As in `OTOCS_poster.ipynb`: `epsilon_0` over the poster's 23 values (cast to
float32, as there), `epsilon_d / g2 = 2, ..., 9`, `n_a = 29`, `n_b = 17`,
`T = 0.2 / kappa`. That is 184 points, one GPU task each, with at most
`N_MAX_GPUS` running at once.

`Wehrl_arr[i, j]` is `S_W(T)`, `rel_arr[i, j]` is `S_W(T) / S_W(0) - 1`, and
`IPR_arr[i, j]` is `IPR(T)`, at `epsilon_0_arr[i]`, `ratio_list[j]`, the layout
of the poster's `lyap_arr`. A failed point is `nan`.

In [ ]:
epsilon_0_list = list(np.concatenate((np.arange(0.2, .6, 0.07), np.arange(0.6, 1.1, 0.03))))
epsilon_0_arr = np.array(epsilon_0_list, dtype=np.float32)
ratio_list = np.arange(2, 10, 1)

T = 2 / kappa
n_time = 50
n_a = 29
n_b = 17

POINTS = [dict(label=f"eps_0 = {eps:.3f}, ed/g2 = {ed:g}", epsilon_0=float(eps), ratio=float(ed))
          for eps in epsilon_0_arr for ed in ratio_list]
COMMON = dict(n_a=n_a, n_b=n_b, T=T, n_time=n_time)

heatmap = run_on_cluster(wehrl_ipr_point, POINTS, COMMON)

shape = (len(epsilon_0_arr), len(ratio_list))
pick = lambda key: np.array([np.nan if "error" in r else r[key] for r in heatmap]).reshape(shape)
Wehrl_arr = pick("S_end")
rel_arr = pick("rel_change")
IPR_arr = pick("ipr_end")
print(f"{np.isnan(Wehrl_arr).sum()} failed point(s) out of {Wehrl_arr.size}")

In [ ]:
d_eps = (epsilon_0_arr[1] - epsilon_0_arr[0]) / 2
extent = (epsilon_0_arr[0] - d_eps, epsilon_0_arr[-1] + d_eps,
          ratio_list[0] - 0.5, ratio_list[-1] + 0.5)

fig, axes = plt.subplots(1, 3, figsize=(16.0, 4.4), constrained_layout=True)
for ax, arr, name in ((axes[0], Wehrl_arr, r"$S_W(T)$"),
                      (axes[1], 100 * rel_arr, r"$S_W(T)/S_W(0)-1$ (%)"),
                      (axes[2], IPR_arr, r"IPR$(T)$")):
    im = ax.imshow(arr.T, origin="lower", aspect="auto", extent=extent)
    ax.set_xlabel(r'$\varepsilon_p$' + ' (rad)')
    ax.set_ylabel(r'$\varepsilon_d / g_2$')
    ax.set_title(name + r', $T=0.2/\kappa_b$, $n_a=$' + str(n_a) + ', ' + r'$n_b=$' + str(n_b))
    fig.colorbar(im, ax=ax)
plt.show()

In [ ]:
import datetime
import json

# A list of rows, one per epsilon_0, one entry per epsilon_d / g2, as the other
# *_array_<date>.json files. "_poster" keeps them apart from wehlr_entropy.ipynb's
# Wehrl_array_<date>.json, which is on another grid. Written next to this
# notebook (or the workspace root, if that is the kernel's working directory)
# under results/.
here = pathlib.Path.cwd()
out_dir = (here if here.name == "Quantum_chaos" else here / "Quantum_chaos") / "results"
out_dir.mkdir(parents=True, exist_ok=True)
date = f"{datetime.date.today():%d_%m_%Y}"

for name, arr in (("Wehrl_array_poster", Wehrl_arr), ("Wehrl_rel_array_poster", rel_arr),
                  ("IPR_array_poster", IPR_arr)):
    out_path = out_dir / f"{name}_{date}.json"
    with open(out_path, "w") as f:
        json.dump(arr.astype(float).tolist(), f)
    print(f"wrote {out_path}")